# Day 55 — A character-level mini-GPT, trained from scratch in numpy

Everything from the last four days in one file: a tokeniser, multi-head attention, a
transformer block, learned positions, Adam, and sampling. No framework and no autograd —
the backward pass is written out by hand and checked against finite differences.

The corpus comes from a small grammar defined in the module, which buys something scraped
text cannot: the entropy of the source is known exactly, so "it learned something" is a
number.

In [1]:
import math

import numpy as np

# ================================================================ 1. the corpus
DET = ['the ', 'a ']


ADJ = ['small ', 'old ', 'red ', 'quiet ', '']


NOUN = ['cat', 'dog', 'bird', 'child', 'robot']


VERB = ['sat', 'slept', 'waited', 'sang', 'ran']


PLACE = ['on the mat', 'in the garden', 'by the river', 'under the tree', 'near the door']


def sentence(rng):
    return (rng.choice(DET) + rng.choice(ADJ) + rng.choice(NOUN) + ' ' +
            rng.choice(VERB) + ' ' + rng.choice(PLACE) + '.\n')


def make_corpus(n_sentences=1200, seed=0):
    rng = np.random.default_rng(seed)
    return ''.join(sentence(rng) for _ in range(n_sentences))


def source_entropy():
    """Bits per character of the generator itself - the floor no model can beat.

    Every sentence is five independent choices, so its information content is the sum
    of the five log2 counts; dividing by the average sentence length gives bits/char.
    """
    bits = sum(math.log2(len(xs)) for xs in (DET, ADJ, NOUN, VERB, PLACE))
    rng = np.random.default_rng(1)
    lengths = [len(sentence(rng)) for _ in range(4000)]
    return bits / float(np.mean(lengths)), bits, float(np.mean(lengths))


def is_grammatical(line):
    """Could the generator have produced this line?"""
    if not line.endswith('.'):
        return False
    body = line[:-1]
    for d in DET:
        if not body.startswith(d):
            continue
        rest = body[len(d):]
        for a in ADJ:
            if not rest.startswith(a):
                continue
            r2 = rest[len(a):]
            for n in NOUN:
                if not r2.startswith(n + ' '):
                    continue
                r3 = r2[len(n) + 1:]
                for v in VERB:
                    if not r3.startswith(v + ' '):
                        continue
                    if r3[len(v) + 1:] in PLACE:
                        return True
    return False


class CharTokenizer:
    """One character, one token - the smallest tokeniser there is."""

    def __init__(self, text):
        self.chars = sorted(set(text))
        self.stoi = {c: i for i, c in enumerate(self.chars)}
        self.itos = {i: c for c, i in self.stoi.items()}

    @property
    def vocab_size(self):
        return len(self.chars)

    def encode(self, s):
        return np.array([self.stoi[c] for c in s], dtype=np.int64)

    def decode(self, ids):
        return ''.join(self.itos[int(i)] for i in ids)


text = make_corpus()
tok = CharTokenizer(text)
data = tok.encode(text)
ent, bits, avg_len = source_entropy()
print(text[:100])
print(f'{len(text):,} characters, vocabulary of {tok.vocab_size}')
print(f'{bits:.3f} bits per sentence over {avg_len:.2f} characters = {ent:.4f} bits/char')
print('round trip:', tok.decode(tok.encode(text)) == text)

a quiet bird slept in the garden.
the small cat sat near the door.
a bird sang near the door.
a quie
38,005 characters, vocabulary of 24
10.288 bits per sentence over 31.56 characters = 0.3260 bits/char
round trip: True


## The model and its hand-written backward pass

Pre-LN blocks of (multi-head attention, MLP), a learned position vector per slot, and a
linear head. The only thing that distinguishes "the loss went down" from "the gradients
are right" is a gradient check: a wrong gradient still trains, just to somewhere else.

In [2]:
def layer_norm(x, g, b, eps=1e-5):
    mu = x.mean(-1, keepdims=True)
    xc = x - mu
    var = (xc ** 2).mean(-1, keepdims=True)
    inv = 1.0 / np.sqrt(var + eps)
    xhat = xc * inv
    return xhat * g + b, (xhat, inv, g)


def layer_norm_backward(dy, cache):
    xhat, inv, g = cache
    d = xhat.shape[-1]
    dg = (dy * xhat).reshape(-1, d).sum(0)
    db = dy.reshape(-1, d).sum(0)
    dxhat = dy * g
    dx = inv / d * (d * dxhat - dxhat.sum(-1, keepdims=True)
                    - xhat * (dxhat * xhat).sum(-1, keepdims=True))
    return dx, dg, db


def gelu(x):
    return 0.5 * x * (1 + np.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * x ** 3)))


def gelu_backward(x, dy):
    c = math.sqrt(2 / math.pi)
    u = c * (x + 0.044715 * x ** 3)
    t = np.tanh(u)
    du = c * (1 + 3 * 0.044715 * x ** 2)
    return dy * (0.5 * (1 + t) + 0.5 * x * (1 - t ** 2) * du)


def softmax(x, axis=-1):
    m = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - m)
    return e / e.sum(axis=axis, keepdims=True)


# ================================================================ 2. the model
def init_params(vocab, d_model=64, n_layer=2, n_head=4, ctx=32, seed=0):
    rng = np.random.default_rng(seed)
    s = 0.02
    p = {'tok': rng.standard_normal((vocab, d_model)) * s,
         'pos': rng.standard_normal((ctx, d_model)) * s,
         'g_f': np.ones(d_model), 'b_f': np.zeros(d_model),
         'head': rng.standard_normal((d_model, vocab)) * s,
         'head_b': np.zeros(vocab)}
    for l in range(n_layer):
        p[f'g1_{l}'] = np.ones(d_model)
        p[f'b1_{l}'] = np.zeros(d_model)
        p[f'g2_{l}'] = np.ones(d_model)
        p[f'b2_{l}'] = np.zeros(d_model)
        p[f'Wqkv_{l}'] = rng.standard_normal((d_model, 3 * d_model)) * s
        p[f'Wo_{l}'] = rng.standard_normal((d_model, d_model)) * s / math.sqrt(2 * n_layer)
        p[f'W1_{l}'] = rng.standard_normal((d_model, 4 * d_model)) * s
        p[f'b1f_{l}'] = np.zeros(4 * d_model)
        p[f'W2_{l}'] = rng.standard_normal((4 * d_model, d_model)) * s / math.sqrt(2 * n_layer)
        p[f'b2f_{l}'] = np.zeros(d_model)
    cfg = {'d_model': d_model, 'n_layer': n_layer, 'n_head': n_head, 'ctx': ctx,
           'vocab': vocab}
    return p, cfg


def param_count(p):
    return sum(v.size for v in p.values())


def forward(p, cfg, idx, causal=True):
    """idx is (B, T) of token ids.  Returns logits (B, T, vocab) and a cache."""
    B, T = idx.shape
    H, d = cfg['n_head'], cfg['d_model']
    dh = d // H
    x = p['tok'][idx] + p['pos'][:T]
    cache = {'idx': idx, 'T': T, 'causal': causal, 'layers': []}
    mask = np.tril(np.ones((T, T), dtype=bool)) if causal else None
    for l in range(cfg['n_layer']):
        h, c1 = layer_norm(x, p[f'g1_{l}'], p[f'b1_{l}'])
        qkv = h @ p[f'Wqkv_{l}']
        q, k, v = np.split(qkv, 3, axis=-1)
        q = q.reshape(B, T, H, dh).transpose(0, 2, 1, 3)
        k = k.reshape(B, T, H, dh).transpose(0, 2, 1, 3)
        v = v.reshape(B, T, H, dh).transpose(0, 2, 1, 3)
        att = q @ k.transpose(0, 1, 3, 2) / math.sqrt(dh)
        if mask is not None:
            att = np.where(mask, att, -np.inf)
        w = softmax(att, axis=-1)
        o = (w @ v).transpose(0, 2, 1, 3).reshape(B, T, d)
        a = o @ p[f'Wo_{l}']
        x = x + a
        h2, c2 = layer_norm(x, p[f'g2_{l}'], p[f'b2_{l}'])
        f1 = h2 @ p[f'W1_{l}'] + p[f'b1f_{l}']
        f2 = gelu(f1)
        f3 = f2 @ p[f'W2_{l}'] + p[f'b2f_{l}']
        x = x + f3
        cache['layers'].append({'c1': c1, 'h': h, 'q': q, 'k': k, 'v': v, 'w': w,
                                'o': o, 'c2': c2, 'h2': h2, 'f1': f1, 'f2': f2})
    xf, cf = layer_norm(x, p['g_f'], p['b_f'])
    logits = xf @ p['head'] + p['head_b']
    cache['cf'] = cf
    cache['xf'] = xf
    return logits, cache


def cross_entropy(logits, targets):
    B, T, V = logits.shape
    flat = logits.reshape(-1, V)
    probs = softmax(flat, axis=-1)
    n = flat.shape[0]
    tgt = targets.reshape(-1)
    loss = -np.log(probs[np.arange(n), tgt] + 1e-12).mean()
    dlogits = probs
    dlogits[np.arange(n), tgt] -= 1.0
    return loss, (dlogits / n).reshape(B, T, V)


def backward(p, cfg, cache, dlogits):
    """The hand-written backward pass: one dictionary of gradients, same keys as p."""
    B, T = cache['idx'].shape
    H, d = cfg['n_head'], cfg['d_model']
    dh = d // H
    g = {k: np.zeros_like(v) for k, v in p.items()}
    g['head'] = cache['xf'].reshape(-1, d).T @ dlogits.reshape(-1, cfg['vocab'])
    g['head_b'] = dlogits.reshape(-1, cfg['vocab']).sum(0)
    dxf = dlogits @ p['head'].T
    dx, g['g_f'], g['b_f'] = layer_norm_backward(dxf, cache['cf'])
    for l in reversed(range(cfg['n_layer'])):
        c = cache['layers'][l]
        # --- MLP branch
        df3 = dx
        g[f'W2_{l}'] = c['f2'].reshape(-1, 4 * d).T @ df3.reshape(-1, d)
        g[f'b2f_{l}'] = df3.reshape(-1, d).sum(0)
        df2 = df3 @ p[f'W2_{l}'].T
        df1 = gelu_backward(c['f1'], df2)
        g[f'W1_{l}'] = c['h2'].reshape(-1, d).T @ df1.reshape(-1, 4 * d)
        g[f'b1f_{l}'] = df1.reshape(-1, 4 * d).sum(0)
        dh2 = df1 @ p[f'W1_{l}'].T
        dx2, g[f'g2_{l}'], g[f'b2_{l}'] = layer_norm_backward(dh2, c['c2'])
        dx = dx + dx2
        # --- attention branch
        da = dx
        g[f'Wo_{l}'] = c['o'].reshape(-1, d).T @ da.reshape(-1, d)
        do = da @ p[f'Wo_{l}'].T
        do = do.reshape(B, T, H, dh).transpose(0, 2, 1, 3)
        dw = do @ c['v'].transpose(0, 1, 3, 2)
        dv = c['w'].transpose(0, 1, 3, 2) @ do
        # softmax backward, row by row
        datt = c['w'] * (dw - (dw * c['w']).sum(-1, keepdims=True))
        datt = datt / math.sqrt(dh)
        dq = datt @ c['k']
        dk = datt.transpose(0, 1, 3, 2) @ c['q']
        def merge(t):
            return t.transpose(0, 2, 1, 3).reshape(B, T, d)
        dqkv = np.concatenate([merge(dq), merge(dk), merge(dv)], axis=-1)
        g[f'Wqkv_{l}'] = c['h'].reshape(-1, d).T @ dqkv.reshape(-1, 3 * d)
        dh1 = dqkv @ p[f'Wqkv_{l}'].T
        dx1, g[f'g1_{l}'], g[f'b1_{l}'] = layer_norm_backward(dh1, c['c1'])
        dx = dx + dx1
    np.add.at(g['tok'], cache['idx'].reshape(-1), dx.reshape(-1, d))
    g['pos'][:T] = dx.sum(0)
    return g


# ================================================================ 3. training
def get_batch(data, ctx, batch, rng):
    i = rng.integers(0, len(data) - ctx - 1, batch)
    x = np.stack([data[j:j + ctx] for j in i])
    y = np.stack([data[j + 1:j + ctx + 1] for j in i])
    return x, y


# ================================================================ 5. gradient check
def gradcheck(seed=0, n=6, eps=1e-5):
    """Finite differences against the hand-written backward pass."""
    text = make_corpus(40, seed=2)
    tok = CharTokenizer(text)
    data = tok.encode(text)
    p, cfg = init_params(tok.vocab_size, d_model=16, n_layer=2, n_head=2, ctx=8, seed=1)
    rng = np.random.default_rng(seed)
    x, y = get_batch(data, cfg['ctx'], 4, rng)
    logits, cache = forward(p, cfg, x)
    loss, dlogits = cross_entropy(logits, y)
    grads = backward(p, cfg, cache, dlogits)
    worst = 0.0
    for k in ['tok', 'pos', 'Wqkv_0', 'Wo_0', 'W1_1', 'g1_0', 'b2f_1', 'head']:
        flat = p[k].ravel()
        gflat = grads[k].ravel()
        for j in rng.integers(0, flat.size, n):
            old = flat[j]
            flat[j] = old + eps
            lp = cross_entropy(forward(p, cfg, x)[0], y)[0]
            flat[j] = old - eps
            lm = cross_entropy(forward(p, cfg, x)[0], y)[0]
            flat[j] = old
            num = (lp - lm) / (2 * eps)
            ana = gflat[j]
            denom = max(1e-8, abs(num) + abs(ana))
            worst = max(worst, abs(num - ana) / denom)
    return worst


p, cfg = init_params(tok.vocab_size)
print(f"d_model {cfg['d_model']}, {cfg['n_layer']} blocks, {cfg['n_head']} heads, "
      f"context {cfg['ctx']}")
print(f'{param_count(p):,} parameters')
print(f'gradient check, worst relative error: {gradcheck():.2e}')

d_model 64, 2 blocks, 4 heads, context 32
104,728 parameters
gradient check, worst relative error: 3.06e-06


## Training

Adam, no schedule, no clipping. The starting loss should be about `ln(vocab)`, which is
what knowing nothing costs.

In [3]:
def train(p, cfg, data, steps=500, batch=16, lr=3e-3, seed=0, causal=True,
          shift=1, log_every=None):
    """Adam, no schedule, no clipping - the smallest loop that works.

    `shift=1` means "predict the next character".  Section 7 sets it to 0.
    """
    rng = np.random.default_rng(seed)
    m = {k: np.zeros_like(v) for k, v in p.items()}
    v = {k: np.zeros_like(x) for k, x in p.items()}
    b1, b2, eps = 0.9, 0.95, 1e-8
    hist = []
    for t in range(1, steps + 1):
        i = rng.integers(0, len(data) - cfg['ctx'] - 1, batch)
        x = np.stack([data[j:j + cfg['ctx']] for j in i])
        y = np.stack([data[j + shift:j + cfg['ctx'] + shift] for j in i])
        logits, cache = forward(p, cfg, x, causal=causal)
        loss, dlogits = cross_entropy(logits, y)
        grads = backward(p, cfg, cache, dlogits)
        for k in p:
            m[k] = b1 * m[k] + (1 - b1) * grads[k]
            v[k] = b2 * v[k] + (1 - b2) * grads[k] ** 2
            mh = m[k] / (1 - b1 ** t)
            vh = v[k] / (1 - b2 ** t)
            p[k] -= lr * mh / (np.sqrt(vh) + eps)
        hist.append(loss)
        if log_every and (t == 1 or t % log_every == 0):
            print(f'  step {t:>5}   loss {loss:.4f}   bits/char {loss / math.log(2):.4f}')
    return hist


def evaluate(p, cfg, data, rng, batches=20, batch=32):
    losses = []
    for _ in range(batches):
        x, y = get_batch(data, cfg['ctx'], batch, rng)
        logits, _ = forward(p, cfg, x)
        losses.append(cross_entropy(logits, y)[0])
    return float(np.mean(losses))


hist = train(p, cfg, data, steps=500, log_every=100)
final = float(np.mean(hist[-50:]))
val = evaluate(p, cfg, CharTokenizer(text).encode(make_corpus(200, seed=99)),
               np.random.default_rng(0))
print()
print(f'training loss {final:.4f} = {final / math.log(2):.4f} bits/char')
print(f'held-out loss {val:.4f} = {val / math.log(2):.4f} bits/char')
print(f'source floor  {ent:.4f} bits/char')

  step     1   loss 3.1724   bits/char 4.5769
  step   100   loss 0.7284   bits/char 1.0508
  step   200   loss 0.3730   bits/char 0.5381
  step   300   loss 0.3515   bits/char 0.5070
  step   400   loss 0.3795   bits/char 0.5474
  step   500   loss 0.3367   bits/char 0.4858

training loss 0.3375 = 0.4869 bits/char
held-out loss 0.3301 = 0.4762 bits/char
source floor  0.3260 bits/char


## Sampling, and a check that is not a vibe

Because the grammar is known, every generated line can be tested: could the generator
have produced it? Temperature moves that number a long way.

In [4]:
# ================================================================ 4. sampling
def sample(p, cfg, tok, prompt='the ', n=200, temperature=1.0, top_k=None, seed=0):
    rng = np.random.default_rng(seed)
    ids = list(tok.encode(prompt))
    for _ in range(n):
        window = np.array(ids[-cfg['ctx']:])[None, :]
        logits, _ = forward(p, cfg, window)
        z = logits[0, -1] / max(temperature, 1e-6)
        if top_k is not None:
            cut = np.sort(z)[-top_k]
            z = np.where(z >= cut, z, -np.inf)
        probs = softmax(z)
        ids.append(int(rng.choice(len(probs), p=probs)))
    return tok.decode(ids)


def grammatical_fraction(text):
    """Of the complete lines in a sample, how many could the generator have made?"""
    lines = [l for l in text.split('\n')[1:-1] if l]
    if not lines:
        return 0.0, 0
    ok = sum(is_grammatical(l) for l in lines)
    return ok / len(lines), len(lines)


for temp, top_k in ((1.0, None), (0.8, 5), (0.2, None)):
    out = sample(p, cfg, tok, n=240, temperature=temp, top_k=top_k, seed=7)
    frac, n = grammatical_fraction(out)
    print(f'temperature {temp}' + (f', top-k {top_k}' if top_k else '') +
          f': {frac * 100:.0f}% of {n} lines grammatical')
    print('   ', out[:90].replace('\n', ' | '))

temperature 1.0: 67% of 6 lines grammatical
    the old bird sat in the garden. | the bird ran under the tree. | a quiet bird sathe garden. | th
temperature 0.8, top-k 5: 50% of 6 lines grammatical
    the mat. | the quiet child ran near the doog ran under the tree. | the old bird waithed in the
temperature 0.2: 100% of 7 lines grammatical
    the garden. | the old robot sang under the tree. | the old robot slept in the garden. | a old ro


## Silent failure #1: training without the causal mask

The unmasked run reaches a much lower loss, and it is cheating — every position can see
the character it is being asked to predict. Nothing in the shapes, the curve or the
gradients says so; it only shows up at generation time, when there is no future to read.

In [5]:
p2, cfg2 = init_params(tok.vocab_size, seed=0)
hist2 = train(p2, cfg2, data, steps=250, causal=False)
print(f'no mask, 250 steps: loss {np.mean(hist2[-50:]):.4f}')
print(f'with mask, same steps: loss {np.mean(hist[200:250]):.4f}')
out2 = sample(p2, cfg2, tok, n=240, temperature=0.8, seed=7)
frac2, n2 = grammatical_fraction(out2)
print(f'the model with the LOWER loss generates {frac2 * 100:.0f}% grammatical lines:')
print('   ', out2[:90].replace('\n', ' | '))

no mask, 250 steps: loss 0.0491
with mask, same steps: loss 0.3824
the model with the LOWER loss generates 0% grammatical lines:
    the e e eeeeeeeeeeeeeeeeeee e ee.e. |  de dedededededededededededededededededededededededede


## Silent failure #2: predicting the character you were just given

Shift the targets by 0 instead of 1 and the loss heads for zero, which is easy to read as
success. The model is learning to copy its input through, and at generation time the
character it is being asked to copy is the one that does not exist yet.

In [6]:
p4, cfg4 = init_params(tok.vocab_size, seed=0)
hist4 = train(p4, cfg4, data, steps=150, shift=0)
print(f'targets shifted by 0: loss {np.mean(hist4[-20:]):.4f}')
print(f'targets shifted by 1: loss {np.mean(hist[130:150]):.4f}')
print('sample:', repr(sample(p4, cfg4, tok, n=60, temperature=0.8, seed=7)))

targets shifted by 0: loss 0.0012
targets shifted by 1: loss 0.4863
sample: 'the                                                             '


## The assertions

In [7]:
# ================================================================ 6. assertions
def check():
    # the grammar checker accepts what the grammar makes and rejects small edits
    rng = np.random.default_rng(5)
    for _ in range(50):
        assert is_grammatical(sentence(rng).strip())
    assert not is_grammatical('the cat sat on the moon.')
    assert not is_grammatical('cat the sat on the mat.')
    assert not is_grammatical('the cat sat on the mat')

    # the tokeniser round-trips
    text = make_corpus(20, seed=3)
    tok = CharTokenizer(text)
    assert tok.decode(tok.encode(text)) == text

    # the hand-written backward pass agrees with finite differences
    assert gradcheck() < 2e-5

    # causal masking really does hide the future: changing a later token cannot
    # change an earlier prediction
    data = tok.encode(text)
    p, cfg = init_params(tok.vocab_size, d_model=16, n_layer=1, n_head=2, ctx=8, seed=4)
    x, _ = get_batch(data, cfg['ctx'], 2, np.random.default_rng(0))
    base, _ = forward(p, cfg, x)
    x2 = x.copy()
    x2[:, -1] = (x2[:, -1] + 1) % cfg['vocab']
    changed, _ = forward(p, cfg, x2)
    assert np.abs(base[:, :-1] - changed[:, :-1]).max() < 1e-12
    nomask, _ = forward(p, cfg, x, causal=False)
    nomask2, _ = forward(p, cfg, x2, causal=False)
    assert np.abs(nomask[:, :-1] - nomask2[:, :-1]).max() > 1e-6
    return True


print('all assertions passed' if check() else 'FAILED')

all assertions passed
